In [1]:
#Currently supports 1 race at a time, will implement vectorized environments later
import pystk2
import random

ImportError: DLL load failed while importing pystk2: The specified module could not be found.

In [ ]:
#Currently using HD graphics for debugging. Will disable graphics later
GraphicsConfig = pystk2.GraphicsConfig.hd()
pystk2.init(GraphicsConfig)

libdecor-gtk-WARNING: Failed to initialize GTK
Failed to load plugin 'libdecor-gtk.so': failed to init
No plugins found, falling back on no decorations

(python:33732): Gtk-WARNING **: 18:28:40.327: gtk_disable_setlocale() must be called before gtk_init()
libdecor-gtk-WARNING: Failed to initialize GTK
Failed to load plugin 'libdecor-gtk.so': failed to init
No plugins found, falling back on no decorations


..:: Antarctica Rendering Engine 2.0 ::..


In [ ]:
WorldState = pystk2.WorldState()

In [ ]:
from collections import deque
import numpy as np

class ProcessState:
    def __init__(self, max_speed=30, map_size=100, track_length=2000):
        self.frame = deque(maxlen=4)    # Window holding last 4 frames
        self.max_speed = max_speed
        self.map_size = map_size
        self.track_length = track_length

    def processObservation(self, obs):
        loc = np.array(obs["location"], dtype=np.float32) / self.map_size
        loc = np.clip(loc, -1.0, 1.0)

        vel = np.array(obs["velocity"], dtype=np.float32) / self.max_speed
        vel = np.clip(vel, -1.0, 1.0)

        front = np.array(obs["front"], dtype=np.float32) / self.map_size
        front = np.clip(front, -1.0, 1.0)

        jump = np.array([1.0 if obs["jumping"] else 0.0], dtype=np.float32)

        rotation = np.array(obs["rotation"], dtype=np.float32)

        dist = np.array([obs.get("distance_down_track", 0.0)], dtype=np.float32) / self.track_length

        state = np.concatenate([loc, vel, front, jump, rotation, dist])

        if len(self.frame) == 0:
            for _ in range(4):
                self.frame.append(state)
        else:
            self.frame.append(state)

        return np.concatenate(self.frame)
    



In [ ]:
# --- STEP 1: THE PPO UPDATE FUNCTION ---
import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork

# Instantiate models with an input dimension of 60
actor_net = ActorNetwork(state_dim=60)
critic_net = CriticNetwork(state_dim=60)

# Set to evaluation mode for simulation
actor_net.eval()
critic_net.eval()

# Initialize Optimizer for both networks
optimizer = optim.Adam([
    {'params': actor_net.parameters(), 'lr': 3e-4},
    {'params': critic_net.parameters(), 'lr': 3e-4}
])

def update_ppo(buffer, epochs=4, gamma=0.99, clip_epsilon=0.2):
    # Extract Data from Buffer
    states = torch.FloatTensor([t['state'] for t in buffer])
    
    # Reconstruct the original Actor format of [Steering, Acceleration].
    actions = torch.FloatTensor([[t['action'][1], t['action'][0]] for t in buffer])
    
    rewards = [t['reward'] for t in buffer]
    values = torch.FloatTensor([t['value'] for t in buffer]).unsqueeze(1)
    old_log_probs = torch.FloatTensor([t['log_prob'] for t in buffer]).unsqueeze(1)
    
    # Calculate Discounted Returns
    returns = []
    discounted_reward = 0
    for reward in reversed(rewards):
        discounted_reward = reward + (gamma * discounted_reward)
        returns.insert(0, discounted_reward)
    returns = torch.FloatTensor(returns).unsqueeze(1)
    
    # Calculate Advantages
    advantages = returns - values
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    # Switch models back to train mode
    actor_net.train()
    critic_net.train()
    
    # PPO Update Loop
    for _ in range(epochs):
        action_dists = actor_net(states)
        new_values = critic_net(states)
        
        new_log_probs = action_dists.log_prob(actions).sum(dim=-1, keepdim=True)
        entropy = action_dists.entropy().sum(dim=-1, keepdim=True).mean()
        
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
        
        actor_loss = -torch.min(surr1, surr2).mean()
        critic_loss = F.mse_loss(new_values, returns)
        
        loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    actor_net.eval()
    critic_net.eval()


In [ ]:
# --- STEP 2: THE EPISODIC WRAPPER & TRIGGER ---
import numpy as np
import pystk2

for episode in range(100):
    buffer = []
    total_episode_reward = 0.0
    
    # --- Nishant's Initialization ---
    config = pystk2.RaceConfig(track='lighthouse', num_kart=1, laps=1)
    config.players[0].controller = pystk2.PlayerConfig.Controller.PLAYER_CONTROL
    race = pystk2.Race(config)
    race.start()
    
    # Track details must be loaded after race start
    track = pystk2.Track()
    track.update()
    track_length = track.length
    max_coordinate = np.max(np.abs(track.path_nodes))
    
    processor = ProcessState(max_speed=30, map_size=max_coordinate, track_length=track_length)
    RaceEnded = False
    
    # --- Nishant's Simulation Loop ---
    for step in range(1000):
        if RaceEnded:
            break
            
        WorldState.update()
        
        kart = WorldState.karts[0]
        obs = {
            "location": kart.location,
            "velocity": kart.velocity,
            "front": kart.front,
            "jumping": kart.jumping,
            "rotation": kart.rotation,
            "distance_down_track": kart.distance_down_track
        }
        
        # Mambo's Data Pipeline
        np_obs = processor.processObservation(obs=obs)
        
        # --- Phase 2 Brain Injection ---
        state_tensor = torch.FloatTensor(np_obs).unsqueeze(0)
        
        with torch.no_grad():
            action_dist = actor_net(state_tensor)
            sampled_action = action_dist.sample()
            state_value = critic_net(state_tensor)
            
        steer_val = torch.clamp(sampled_action[0, 0], min=-1.0, max=1.0).item()
        accel_val = torch.clamp(sampled_action[0, 1], min=0.0, max=1.0).item()
        
        action = pystk2.Action()
        action.steer = steer_val
        action.acceleration = accel_val
        
        # Step the environment
        RaceEnded = race.step(action)
        
        # Reward Calculation
        vel_x, vel_y, vel_z = obs['velocity']
        speed = (vel_x**2 + vel_y**2 + vel_z**2)**0.5
        reward = speed * 0.1
        
        if obs.get('distance_down_track', 0.0) < 0:
            reward -= 10.0
            
        total_episode_reward += reward
        
        # Buffer Appending
        transition = {
            "state": np_obs,
            "action": np.array([action.acceleration, action.steer, 0.0, 0.0, 0.0], dtype=np.float32),
            "reward": reward,
            "value": state_value.item(),
            "log_prob": action_dist.log_prob(sampled_action).sum(dim=-1).item(),
            "done": RaceEnded
        }
        buffer.append(transition)
        
    # --- END OF EPISODE TRIGGER ---
    print(f"Episode: {episode + 1}/100 | Total Reward: {total_episode_reward:.2f} | Buffer Size: {len(buffer)}")
    
    # Trigger the Brain Transplant!
    update_ppo(buffer)
    
    # Critical Cleanup
    race.stop()
    del race

pystk2.clean()
